# ДЗ 1

## Бэггинг и бустинг
Для датасета creditcardfraud (мошенничество с кредитными картами, 0 - нет мошенничества, 1 - есть)
- подобрать параметры с помощью GridSearchCV
- вывести и выбрать наилучшие
- обучить бэггинг и бустинг
- сравнить модели (вывести информацию по метрикам)

Для выполнения можно воспользоваться кодом с семинара. Код для загрузки датасета ниже:

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

In [2]:
import pandas as pd
df = pd.read_csv('./creditcard.csv')

In [3]:
df.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 284807 entries, 0 to 284806
Data columns (total 31 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   Time    284807 non-null  float64
 1   V1      284807 non-null  float64
 2   V2      284807 non-null  float64
 3   V3      284807 non-null  float64
 4   V4      284807 non-null  float64
 5   V5      284807 non-null  float64
 6   V6      284807 non-null  float64
 7   V7      284807 non-null  float64
 8   V8      284807 non-null  float64
 9   V9      284807 non-null  float64
 10  V10     284807 non-null  float64
 11  V11     284807 non-null  float64
 12  V12     284807 non-null  float64
 13  V13     284807 non-null  float64
 14  V14     284807 non-null  float64
 15  V15     284807 non-null  float64
 16  V16     284807 non-null  float64
 17  V17     284807 non-null  float64
 18  V18     284807 non-null  float64
 19  V19     284807 non-null  float64
 20  V20     284807 non-null  float64
 21  V21     28

In [5]:
df.isnull().sum().sum() # Проверка на пропущенные значения

0

In [6]:
print('Распределение классов:')
print(df['Class'].value_counts())

Распределение классов:
Class
0    284315
1       492
Name: count, dtype: int64


In [7]:
data = df.copy()

X = data.drop('Class', axis=1)
y = data['Class'] # 0 - нет мошенничества, 1 - есть

X_train, X_test, y_train, y_test = train_test_split( # Для обучения и оценки моделей данные разделены на обучающую и тестовую выборки в соотношении 80/20
    X, y, test_size=0.2, random_state=42, stratify=y # stratify=y, чтобы сохранить исходное распределение классов в обеих выборках
)

print(f'Train: {X_train.shape}, Test: {X_test.shape}')
print(f'Доля миноритарного класса в Train: {np.mean(y_train == 1):.2%}')

Train: (227845, 30), Test: (56962, 30)
Доля миноритарного класса в Train: 0.17%


### Бэггинг

In [10]:
param_grid_rf = {
    'clf__n_estimators': [50, 100, 200],     # Число деревьев
    'clf__max_depth': [5, 10, 15],           # Глубина
    'clf__min_samples_leaf': [1, 2, 4]       # Мин. объектов в листе
}

pipe_rf = Pipeline([
    ('clf', RandomForestClassifier(class_weight='balanced', random_state=42)) # Для реализации бэггинга используется RandomForestClassifier
])

cv_rf = StratifiedKFold(
    n_splits=3, # делим обучающую выборку на 3 части
    shuffle=True, # перемешиваем данные
    random_state=42)

# GridSearchCV
grid_search_rf = GridSearchCV(
    pipe_rf, param_grid_rf, cv=cv_rf, scoring='roc_auc', n_jobs=-1, verbose=2 # roc-auc - из-за сильного дисбаланса классов
)

grid_search_rf.fit(X_train, y_train)

print(f'Лучшие параметры: {grid_search_rf.best_params_}')
print(f'Лучший ROC-AUC на CV: {grid_search_rf.best_score_:.4f}')

Fitting 3 folds for each of 27 candidates, totalling 81 fits
Лучшие параметры: {'clf__max_depth': 5, 'clf__min_samples_leaf': 2, 'clf__n_estimators': 200}
Лучший ROC-AUC на CV: 0.9810


In [11]:
best_rf = grid_search_rf.best_estimator_
y_pred_rf = best_rf.predict(X_test)

### Бустинг

In [12]:
param_grid_gb = {
    'clf__n_estimators': [50, 100, 200],    # Число деревьев
    'clf__learning_rate': [0.05, 0.1, 0.2],  # Скорость обучения
    'clf__max_depth': [2, 3, 4]        # Глубина
}

pipe_gb = Pipeline([
    ('clf', GradientBoostingClassifier(random_state=42)) # Для реализации бустинга используется GradientBoostingClassifier
])

cv_gb = StratifiedKFold(
    n_splits=3,   # делим обучающую выборку на 3 части
    shuffle=True, # перемешиваем данные
    random_state=42)

# GridSearchCV
grid_search_gb = GridSearchCV(
    pipe_gb, param_grid_gb, cv=cv_gb, scoring='roc_auc', n_jobs=-1, verbose=2
)

grid_search_gb.fit(X_train, y_train)

print(f'Лучшие параметры: {grid_search_gb.best_params_}')
print(f'Лучший ROC-AUC на CV: {grid_search_gb.best_score_:.4f}')

Fitting 3 folds for each of 27 candidates, totalling 81 fits
Лучшие параметры: {'clf__learning_rate': 0.05, 'clf__max_depth': 4, 'clf__n_estimators': 200}
Лучший ROC-AUC на CV: 0.8982


In [13]:
best_gb = grid_search_gb.best_estimator_
y_pred_gb = best_gb.predict(X_test)

### Результаты

| Модель | Лучшие параметры | ROC-AUC на CV |
|---|---|---:|
| Random Forest | `max_depth=5`, `min_samples_leaf=2`, `n_estimators=200` | 0.9810 |
| Gradient Boosting | `learning_rate=0.05`, `max_depth=4`, `n_estimators=200` | 0.8982 |

In [18]:
cm_rf = confusion_matrix(y_test, y_pred_rf)
cm_rf

array([[56717,   147],
       [   11,    87]], dtype=int64)

Найдено мошеннических транзакций: 87

Пропущено мошеннических транзакций: 11	

In [19]:
cm_gb = confusion_matrix(y_test, y_pred_gb)
cm_gb

array([[56851,    13],
       [   32,    66]], dtype=int64)

Найдено мошеннических транзакций: 66

Пропущено мошеннических транзакций: 32

In [24]:
print('Random Forest')
print(classification_report(y_test, y_pred_rf))

Random Forest
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.37      0.89      0.52        98

    accuracy                           1.00     56962
   macro avg       0.69      0.94      0.76     56962
weighted avg       1.00      1.00      1.00     56962



In [25]:
print('Gradient Boosting')
print(classification_report(y_test, y_pred_gb))

Gradient Boosting
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.84      0.67      0.75        98

    accuracy                           1.00     56962
   macro avg       0.92      0.84      0.87     56962
weighted avg       1.00      1.00      1.00     56962

